# Занятие 10. Подготовка проекта с базой данных к экзамену

## Цель занятия

Это финальное занятие блока по базам данных.

На прошлых занятиях мы изучили:

- что такое база данных;
- SQLite и `sqlite3`;
- создание таблиц;
- `INSERT`, `SELECT`, `UPDATE`, `DELETE`;
- `WHERE`, `ORDER BY`, `LIMIT`;
- типы данных;
- связи таблиц;
- `FOREIGN KEY`;
- `JOIN`;
- SQL-аналитику;
- Python-функции для работы с БД;
- мини-проект с sqlite3.

Сегодня задача другая:

> подготовить проект к экзамену так, чтобы его можно было уверенно показать преподавателю.

---

## Что значит “проект готов к экзамену”

Проект готов, если студент может показать:

1. тему проекта;
2. структуру проекта;
3. базу данных;
4. таблицы;
5. SQL-запросы;
6. Python-функции;
7. запуск проекта;
8. результат работы;
9. GitHub-репозиторий;
10. краткое объяснение, зачем проект нужен.

---

## Что должен уметь студент на защите

Студент должен уметь сказать:

- какую проблему решает проект;
- какие данные хранятся в БД;
- какие таблицы созданы;
- какие SQL-запросы используются;
- как Python подключается к SQLite;
- как проект запускается;
- какие результаты выводятся;
- что лежит в GitHub.

---

## Связь с реальной жизнью

В реальной разработке важно не только написать код, но и подготовить проект к показу:

- заказчику;
- преподавателю;
- работодателю;
- команде;
- пользователю.

Если проект нельзя запустить и объяснить, он выглядит незавершённым.

---

## Связь с дипломом

Этот урок — репетиция дипломной защиты.

Для диплома студенту нужно будет:

- показать структуру проекта;
- объяснить архитектуру;
- показать базу данных;
- показать SQL;
- показать результат;
- показать GitHub;
- ответить на вопросы.

---

## Связь с GitHub

На этом занятии студент приводит репозиторий в порядок:

- проверяет `main`;
- проверяет ветку занятия;
- делает commit;
- делает push;
- открывает Pull Request;
- делает merge;
- проверяет, что в `main` лежит стабильная версия.

Рекомендуемая ветка:

```bash
git checkout main
git pull origin main
git checkout -b feature/lesson-10-exam-ready
```

Commit:

```bash
git add .
git commit -m "docs: prepare sqlite project for exam"
git push -u origin feature/lesson-10-exam-ready
```

## Ячейка 1. Импорт sqlite3 и создание подключения

В финальном проекте мы ещё раз собираем рабочий пример, который студент может показать на экзамене.

Тема solved-примера: **учебный трекер проектов студентов**.

In [ ]:
import sqlite3

def get_connection(db_name="exam_ready_project.db"):
    return sqlite3.connect(db_name)

connection = get_connection()
cursor = connection.cursor()

print("База данных подключена")
assert connection is not None

## Ячейка 2. Создаём таблицу student_projects

Таблица хранит учебные проекты студентов.

Поля:
- `id`;
- `student_name`;
- `project_topic`;
- `status`;
- `score`;
- `comment`.

In [ ]:
def create_projects_table(connection):
    cursor = connection.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS student_projects (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        student_name TEXT,
        project_topic TEXT,
        status TEXT,
        score REAL,
        comment TEXT
    )
    """)
    connection.commit()

create_projects_table(connection)

print("Таблица student_projects создана")

## Ячейка 3. Очищаем таблицу и добавляем демонстрационные данные

Очищаем таблицу перед повторным запуском, чтобы данные не дублировались.

In [ ]:
def clear_projects(connection):
    cursor = connection.cursor()
    cursor.execute("DELETE FROM student_projects")
    connection.commit()

def add_project(connection, student_name, project_topic, status, score, comment):
    cursor = connection.cursor()
    cursor.execute(
        """
        INSERT INTO student_projects (student_name, project_topic, status, score, comment)
        VALUES (?, ?, ?, ?, ?)
        """,
        (student_name, project_topic, status, score, comment)
    )
    connection.commit()

clear_projects(connection)

add_project(connection, "Анна", "HR Resume Tracker", "ready", 90.0, "Хорошая структура БД")
add_project(connection, "Иван", "Nutrition Analytics", "in_progress", 78.0, "Нужно доработать отчёт")
add_project(connection, "Ольга", "Sport Tracker", "ready", 88.5, "Есть аналитика")
add_project(connection, "Павел", "Shop Analytics", "review", 82.0, "Нужно проверить JOIN")
add_project(connection, "Мария", "Learning Progress DB", "ready", 91.0, "Проект готов к показу")

print("Данные добавлены")

## Ячейка 4. Проверяем все записи

Функция `get_all_projects` показывает все проекты.

На экзамене студент должен уметь показать содержимое таблицы.

In [ ]:
def get_all_projects(connection):
    cursor = connection.cursor()
    cursor.execute("SELECT * FROM student_projects")
    return cursor.fetchall()

projects = get_all_projects(connection)

for project in projects:
    print(project)

assert len(projects) == 5

## Ячейка 5. Фильтр по статусу

Покажем только проекты со статусом `ready`.

Это пример полезного SQL-запроса для экзамена.

In [ ]:
def find_projects_by_status(connection, status):
    cursor = connection.cursor()
    cursor.execute(
        "SELECT * FROM student_projects WHERE status = ?",
        (status,)
    )
    return cursor.fetchall()

ready_projects = find_projects_by_status(connection, "ready")

for project in ready_projects:
    print(project)

assert len(ready_projects) >= 1

## Ячейка 6. Аналитика: средний score

Покажем, что SQL умеет считать аналитику.

In [ ]:
def get_average_score(connection):
    cursor = connection.cursor()
    cursor.execute("SELECT AVG(score) FROM student_projects")
    return cursor.fetchone()[0]

avg_score = get_average_score(connection)

print("Средний score:", round(avg_score, 2))

assert avg_score > 0

## Ячейка 7. Аналитика по статусам

Считаем количество проектов по каждому статусу.

Это хороший пример отчёта.

In [ ]:
def get_status_report(connection):
    cursor = connection.cursor()
    cursor.execute("""
    SELECT status, COUNT(*)
    FROM student_projects
    GROUP BY status
    ORDER BY COUNT(*) DESC
    """)
    return cursor.fetchall()

status_report = get_status_report(connection)

for row in status_report:
    print(row)

assert len(status_report) >= 1

## Ячейка 8. Топ проектов по score

Покажем лучшие проекты.

In [ ]:
def get_top_projects(connection, limit=3):
    cursor = connection.cursor()
    cursor.execute(
        """
        SELECT student_name, project_topic, status, score
        FROM student_projects
        ORDER BY score DESC
        LIMIT ?
        """,
        (limit,)
    )
    return cursor.fetchall()

top_projects = get_top_projects(connection, 3)

for project in top_projects:
    print(project)

assert len(top_projects) == 3

## Ячейка 9. Генерируем текст для README

README нужен для GitHub и экзамена.

Он кратко объясняет:
- что это за проект;
- как его запустить;
- что он умеет.

In [ ]:
readme_text = """# SQLite Exam Project

## Описание
Это учебный проект с базой данных SQLite.

## Что умеет проект
- создаёт таблицу проектов студентов;
- добавляет записи;
- фильтрует данные по статусу;
- считает средний score;
- строит отчёт по статусам;
- показывает топ проектов.

## Запуск
python src/main.py

## Используемые технологии
- Python
- sqlite3
- SQL
- GitHub
"""

print(readme_text)

assert "sqlite3" in readme_text

## Ячейка 10. Финальный сценарий защиты

Собираем итоговый вывод, который студент может показать преподавателю.

In [ ]:
def show_exam_demo(connection):
    print("=== Демонстрация проекта к экзамену ===")
    print()

    print("1. Все проекты:")
    for project in get_all_projects(connection):
        print(project)

    print("\n2. Готовые проекты:")
    for project in find_projects_by_status(connection, "ready"):
        print(project)

    print("\n3. Средний score:")
    print(round(get_average_score(connection), 2))

    print("\n4. Отчёт по статусам:")
    for row in get_status_report(connection):
        print(row)

    print("\n5. Топ-3 проекта:")
    for project in get_top_projects(connection, 3):
        print(project)

show_exam_demo(connection)

connection.close()

print("\nПроект готов к экзамену. Соединение закрыто.")

# Итог занятия 10

Сегодня вы подготовили проект к экзамену.

Студент должен показать:

- базу данных;
- таблицу;
- SQL-запросы;
- Python-функции;
- аналитику;
- запуск проекта;
- GitHub;
- README;
- краткий рассказ о проекте.

## Что переносить в VS Code

```text
src/main.py
src/db/connection.py
src/db/create_tables.py
src/db/queries.py
src/db/reports.py
README.md
data/
```

## Что коммитить в GitHub

```bash
git add .
git commit -m "docs: prepare sqlite project for exam"
git push -u origin feature/lesson-10-exam-ready
```

После этого:
- открыть Pull Request;
- сделать merge в `main`;
- проверить запуск из `main`.